# Metrics Plotter — Producer & Consumer Applications Only

This notebook scans one or more folders for CSV/Parquet metric files, keeps **only** rows that mention
`producer_application` or `consumer_application`, auto-detects timestamp/metric/value columns, and produces
clearly labeled time-series plots. All figures are saved under a single timestamped directory with two subfolders:
`producer_application/` and `consumer_application/`.

**Notes**
- Uses only `matplotlib` (no seaborn) and one plot per figure.
- If a valid timestamp column is not found, the plot is drawn against simple index order.
- Change `INPUT_DIRS` in the *Config* cell to point to your metric folders.


In [8]:
# =====================
# Config
# =====================
import os
import re
import zipfile
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- DIRECTORIES TO SEARCH (edit this to your paths) ----
Home = "/Users/soheila/Desktop/RealTime-Streaming-Pipeline/"
INPUT_DIRS = [
    os.path.join(Home, "results/20250908_140639/merge/merge-sts-0_merge-metrics/2025-09-08_00-57-10/"),
]

# ---- METRIC FILTER: keep ONLY these application metrics ----
METRIC_KEYWORDS = ["producer_application", "consumer_application"]

# ---- OUTPUT ROOT (timestamped) ----
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# Use pathlib for proper path handling
PLOTS_ROOT = Path(Home) / "results" / "plots" / f"run_{RUN_STAMP}"
(PLOTS_ROOT / "producer_application").mkdir(parents=True, exist_ok=True)
(PLOTS_ROOT / "consumer_application").mkdir(parents=True, exist_ok=True)

print(f"[Init] Output directory: {PLOTS_ROOT}")


[Init] Output directory: /Users/soheila/Desktop/RealTime-Streaming-Pipeline/results/plots/run_20250909_152713


In [9]:
# =====================
# Helpers
# =====================
from typing import List
import pandas as pd

CSV_EXT = ".csv"
PARQUET_EXT = ".parquet"

def list_metric_files(dirs: List[str], exts=(CSV_EXT, PARQUET_EXT)) -> List[Path]:
    """Recursively list metric files with the given extensions."""
    out = []
    for d in dirs:
        d = Path(d)
        if not d.exists():
            continue
        for root, _, files in os.walk(d):
            for f in files:
                if f.lower().endswith(exts):
                    out.append(Path(root) / f)
    return sorted(out)

def read_any(path: Path) -> pd.DataFrame:
    """Read CSV or Parquet into a DataFrame, return empty DataFrame on failure."""
    try:
        if path.suffix.lower() == CSV_EXT:
            return pd.read_csv(path)
        elif path.suffix.lower() == PARQUET_EXT:
            try:
                return pd.read_parquet(path, engine="pyarrow")
            except Exception:
                return pd.read_parquet(path, engine="fastparquet")
        return pd.DataFrame()
    except Exception as e:
        print(f"[WARN] Failed to read {path}: {e}")
        return pd.DataFrame()

def lowercase_columns(df: pd.DataFrame) -> pd.DataFrame:
    c = [str(col).strip().lower() for col in df.columns]
    return df.rename(columns=dict(zip(df.columns, c)))

def find_timestamp_col(df: pd.DataFrame):
    """Heuristically find a time-like column."""
    if "timestamp" in df.columns:
        return "timestamp"
    candidates = [c for c in df.columns if any(k in c for k in ["timestamp", "time", "ts", "date"])]
    return candidates[0] if candidates else None

def find_metric_name_col(df: pd.DataFrame):
    """Find a text column that likely holds the metric name/identifier."""
    for c in ["metric", "metric_name", "name", "id", "label"]:
        if c in df.columns and df[c].dtype == object:
            return c
    objs = [c for c in df.columns if df[c].dtype == object]
    return objs[0] if objs else None

def find_value_col(df: pd.DataFrame, exclude=None):
    """Find a numeric 'value' column; prefer common names then first numeric."""
    exclude = set(exclude or [])
    if "value" in df.columns and pd.api.types.is_numeric_dtype(df["value"]):
        return "value"
    for c in ["val", "count", "gauge", "measurement", "amount"]:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            return c
    numeric_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
    return numeric_cols[0] if numeric_cols else None

def filter_keywords_rows(df: pd.DataFrame, keywords):
    """Keep rows where ANY string column contains any of the keywords (case-insensitive)."""
    if df.empty:
        return df
    str_cols = [c for c in df.columns if df[c].dtype == object]
    if not str_cols:
        return pd.DataFrame(columns=df.columns)
    mask = np.zeros(len(df), dtype=bool)
    for c in str_cols:
        col = df[c].astype(str).str.lower()
        for kw in keywords:
            mask |= col.str.contains(re.escape(kw.lower()), na=False)
    return df[mask].copy()

def to_datetime_if_possible(series: pd.Series):
    """Convert to datetime if feasible; otherwise return as-is."""
    try:
        if pd.api.types.is_numeric_dtype(series):
            dt = pd.to_datetime(series, unit="s", origin="unix", errors="coerce")
            if dt.notna().sum() >= max(3, int(0.5 * len(series))):
                return dt
            dt = pd.to_datetime(series, unit="ms", origin="unix", errors="coerce")
            if dt.notna().sum() >= max(3, int(0.5 * len(series))):
                return dt
        return pd.to_datetime(series, errors="coerce")
    except Exception:
        return series

def sanitize_filename(s: str) -> str:
    import re
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(s))[:150]

def plot_timeseries(x, y, title, xlabel, ylabel, out_path: Path):
    """Time-series plot (single figure, no explicit colors/styles)."""
    plt.figure()
    plt.plot(x, y)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

def plot_index_series(y, title, ylabel, out_path: Path):
    """Index vs value when no time column is available."""
    plt.figure()
    plt.plot(range(len(y)), y)
    plt.title(title)
    plt.xlabel("Index")
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


In [10]:
# =====================
# Main: Scan → Filter → Plot
# =====================
all_files = list_metric_files(INPUT_DIRS)
print(f"[Scan] Found {len(all_files)} files.")

plot_count = 0
kept_summary = []

for fpath in all_files:
    df = read_any(fpath)
    if df.empty:
        continue

    df = lowercase_columns(df)
    df["__source_file"] = str(fpath)

    # Keep ONLY producer/consumer application rows
    df_keep = filter_keywords_rows(df, METRIC_KEYWORDS)
    if df_keep.empty:
        continue

    # Detect key columns
    tcol = find_timestamp_col(df_keep)
    mcol = find_metric_name_col(df_keep)
    vcol = find_value_col(df_keep, exclude=[c for c in [tcol, mcol, "__source_file"] if c])
    if vcol is None:
        print(f"[Skip] No numeric value column in {Path(fpath).name}")
        continue

    # Convert time if present
    if tcol:
        df_keep[tcol] = to_datetime_if_possible(df_keep[tcol])

    # Label each row as producer or consumer
    def label_side(row) -> str:
        txt = " ".join([str(row[c]) for c in df_keep.columns if df_keep[c].dtype == object]).lower()
        if "producer_application" in txt:
            return "producer_application"
        if "consumer_application" in txt:
            return "consumer_application"
        return "unknown"

    df_keep["__side"] = df_keep.apply(label_side, axis=1)
    df_keep = df_keep[df_keep["__side"].isin(["producer_application", "consumer_application"])]
    if df_keep.empty:
        continue

    # If no explicit metric-name column, synthesize a short one
    if mcol is None:
        obj_cols = [c for c in df_keep.columns if df_keep[c].dtype == object and c not in ["__side", "__source_file"]]
        mcol = obj_cols[0] if obj_cols else "__metric_name"
        if mcol == "__metric_name":
            df_keep[mcol] = "metric"

    # Group and plot
    for side_name, g_side in df_keep.groupby("__side"):
        for metric_name, g in g_side.groupby(mcol):
            metric_label = str(metric_name)
            metric_label_short = metric_label if len(metric_label) <= 80 else metric_label[:77] + "..."
            base_filename = sanitize_filename(f"{side_name}__{metric_label}") + ".png"
            out_dir = PLOTS_ROOT / side_name
            out_path = out_dir / base_filename

            title = f"{side_name.replace('_', ' ').title()} — {metric_label_short}\nSource file: {Path(fpath).name}"
            ylabel = f"Value ({vcol})"

            if tcol and g[tcol].notna().any():
                g_sorted = g.sort_values(by=tcol)
                plot_timeseries(
                    x=g_sorted[tcol],
                    y=g_sorted[vcol].astype(float, errors='ignore'),
                    title=title,
                    xlabel="Time",
                    ylabel=ylabel,
                    out_path=out_path,
                )
            else:
                plot_index_series(
                    y=g[vcol].astype(float, errors='ignore'),
                    title=title + " (Index order)",
                    ylabel=ylabel,
                    out_path=out_path,
                )

            plot_count += 1
            kept_summary.append({
                "source_file": str(fpath),
                "side": side_name,
                "metric_name": metric_label,
                "value_col": vcol,
                "time_col": tcol or "",
                "output_path": str(out_path),
            })

print(f"[Done] Saved {plot_count} plots into {PLOTS_ROOT}")

# Save a summary CSV
summary_df = pd.DataFrame(kept_summary)
summary_csv = PLOTS_ROOT / "plot_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"[Summary] {len(summary_df)} entries -> {summary_csv}")


[Scan] Found 4 files.
[Done] Saved 8 plots into /Users/soheila/Desktop/RealTime-Streaming-Pipeline/results/plots/run_20250909_152713
[Summary] 8 entries -> /Users/soheila/Desktop/RealTime-Streaming-Pipeline/results/plots/run_20250909_152713/plot_summary.csv


In [11]:
# =====================
# Zip the plots directory (optional)
# =====================
zip_path = PLOTS_ROOT.with_suffix(".zip")
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(PLOTS_ROOT):
        for f in files:
            fp = Path(root) / f
            arcname = fp.relative_to(PLOTS_ROOT.parent)
            zf.write(fp, arcname)

print(f"[Zip] Created: {zip_path}")


[Zip] Created: /Users/soheila/Desktop/RealTime-Streaming-Pipeline/results/plots/run_20250909_152713.zip
